# Design exercise D1 · Late deliveries — Part B

RomaEats' care team phones **400 customers for every 4,000 orders** before
their order turns late. Your job: give each of 4,000 new orders a score, so
that the 400 highest-scored are as many late orders as possible.

You have about 15 minutes. The cells below load the data, define the metric,
cross-validate a baseline and write the file. Make **your** design choices in
the cell marked *Your model*, compare them on the folds, then hand in.

**Scoring.** Your 400 highest scores are compared with the real outcomes,
which only the grader has. The precision among them gives the points: 0 at the
level of a random ranking (9%), 10 at the reference model's 40% or above,
linear in between.

> **One attempt.** Only the **first** file you turn in is scored. If you
> unsubmit and turn in another file, it is not scored: you keep the score of
> the first. Choose your model with the notebook's own validation, check the file with the last
> cell, and only then turn it in.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

BASE = "https://raw.githubusercontent.com/fabsilvestri/AIML-Course/main/design/d01/"
train = pd.read_csv(BASE + "train.csv")
new = pd.read_csv(BASE + "test.csv")          # the 4,000 orders to score

assert train.shape == (6000, 11) and new.shape == (4000, 10)
assert train["late"].isin([0, 1]).all()
print(f"{len(train):,} past orders, {train['late'].mean():.1%} late; "
      f"{len(new):,} new orders to score")
train.head()

The metric is the one the brief fixes: the precision among the highest-scored
tenth of the orders. Cross-validation uses stratified folds, so every fold
keeps the 8.7% of late orders.

In [ ]:
def precision_top(y, scores, share=0.10):
    """Share of late orders among the `share` highest scores."""
    k = int(round(share * len(y)))
    top = np.argsort(-np.asarray(scores), kind="stable")[:k]
    return np.asarray(y)[top].mean()

X = train.drop(columns=["order_id", "late"])
y = train["late"].to_numpy()
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def cv_precision(model):
    """Out-of-fold scores from cross_val_predict, then the metric."""
    scores = cross_val_predict(model, X, y, cv=folds,
                               method="predict_proba")[:, 1]
    return precision_top(y, scores)

A baseline, to beat: logistic regression on the numerical columns as they
come, with the blanks imputed inside the pipeline.

In [ ]:
numerical = ["hour", "distance_km", "prep_time_avg_min", "courier_load",
             "rain", "items", "basket_eur"]

baseline = make_pipeline(
    make_column_transformer(
        (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()),
         numerical)),
    LogisticRegression())

print(f"baseline: {cv_precision(baseline):.3f}")

## Your model

Build your design here, and compare it with the baseline on the folds. Every
choice — which columns, how each is encoded, which model, which
hyperparameters — is yours, and is judged by `cv_precision`.

In [ ]:
model = make_pipeline(
    make_column_transformer(
        (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()),
         numerical)),
    LogisticRegression())          # replace with your design

print(f"your model: {cv_precision(model):.3f}")

## Hand in

Fit the chosen model on all 6,000 orders, score the 4,000 new ones, and check
the file before you write it.

In [ ]:
model.fit(X, y)
score = model.predict_proba(new.drop(columns=["order_id"]))[:, 1]

out = pd.DataFrame({"order_id": new["order_id"], "score": score})
assert len(out) == 4000 and out["order_id"].is_unique
assert out["score"].notna().all()
assert out["score"].nunique() > 400, "too many ties to choose 400 calls"
out.to_csv("predictions.csv", index=False)
print("wrote predictions.csv:", len(out), "rows")

In Colab, download it (the folder icon on the left, or run the next cell),
then in Classroom open the assignment, **Add → File**, choose
`predictions.csv`, and **Turn in**. The grader marks it within the hour.
**Only this first file counts**: a second one is not scored.

In [ ]:
try:
    from google.colab import files
    files.download("predictions.csv")
except ImportError:
    print("Not in Colab: predictions.csv is in the working directory.")